In [ ]:
# =====================================================
# 0. 패키지 설치
# =====================================================
# 처음 한 번만 실행
# !pip install webdriver-manager

In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

import re
import time
import pandas as pd

from datetime import datetime, timedelta
from tqdm import tqdm
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
# =====================================================
# 2. 기본 설정
# =====================================================

# 38R 기준일
MATCH_38R = datetime(2026, 5, 25, 0, 0, 0)

# 카페 수집 기간: 38R 전후 1주일(토트넘 기준 1주일 게시글 214개, 데이터 2500개 수집)
START_DATE = MATCH_38R - timedelta(days=7)
END_DATE = MATCH_38R + timedelta(days=7, hours=23, minutes=59, seconds=59)

# 날짜에 연도가 없는 경우 사용할 기준 연도
BASE_YEAR = 2026

# 카페 검색/게시판 URL
# 네이버 카페에서 게시판 URL을 넣으세요. ex)토트넘 게시판 
BASE_URL = "https://cafe.naver.com/f-e/cafes/29267144/menus/12" # https://cafe.naver.com/f-e/cafes/29267144/menus/12?viewType=L 이것 중 맨 뒤에 ?viewType=L 같은 쿼리는 삭제 !!

# 몇 페이지까지 탐색할지
MAX_PAGES = 80

# 요청 간 대기 시간
SLEEP_SEC = 1.0

# 최종 저장 파일명 팀 명에 맞게 수정
OUTPUT_CSV = "tottenham_cafe_38R_before_after.csv"
OUTPUT_XLSX = "tottenham_cafe_38R_before_after.xlsx"

In [ ]:
# =====================================================
# 3. 텍스트 전처리 함수
# =====================================================

def clean_text(text):
    text = str(text)

    # 줄바꿈 제거
    text = text.replace("\n", " ").replace("\r", " ")

    # URL 제거
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # HTML entity 느낌 제거
    text = re.sub(r"&[a-zA-Z]+;", " ", text)

    # 특수문자 일부 정리
    text = re.sub(r"[^ㄱ-ㅎ가-힣a-zA-Z0-9\s.,!?ㅋㅋㅎㅎㅠㅜ]", " ", text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_period(dt):
    if pd.isna(dt):
        return "unknown"

    if dt < MATCH_38R:
        return "before_38R"
    else:
        return "after_38R"


def parse_cafe_date(date_text):
    """
    네이버 카페 날짜 문자열을 datetime으로 변환.
    예:
    - 2025.05.25. 00:12
    - 2025.05.25.
    - 05.25. 00:12
    - 05.25.
    """

    if date_text is None:
        return None

    text = str(date_text).strip()
    text = re.sub(r"\s+", " ", text)

    # 방금, 몇 분 전, 어제 등은 정확한 날짜 계산이 애매하므로 None 처리
    if any(x in text for x in ["방금", "분 전", "시간 전", "어제"]):
        return None

    patterns = [
        "%Y.%m.%d. %H:%M",
        "%Y.%m.%d. %H:%M:%S",
        "%Y.%m.%d.",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%d",
        "%m.%d. %H:%M",
        "%m.%d.",
    ]

    for p in patterns:
        try:
            dt = datetime.strptime(text, p)

            # 연도가 없는 패턴이면 BASE_YEAR 부여
            if p.startswith("%m"):
                dt = dt.replace(year=BASE_YEAR)

            return dt

        except:
            pass

    return None

In [ ]:
# =====================================================
# 4. Selenium 드라이버 실행
# =====================================================

chrome_options = Options()

# 브라우저 창 열기
chrome_options.add_argument("--start-maximized")

# 자동화 탐지 최소화 옵션
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

driver.get("https://nid.naver.com/nidlogin.login")

print("네이버 로그인을 수동으로 완료한 뒤, 아래 input 칸에서 Enter를 누르세요.")
input("로그인 완료 후 Enter 입력: ")

In [ ]:
# =====================================================
# 5. 유틸 함수
# =====================================================

def switch_to_cafe_iframe():
    """
    네이버 카페 본문/목록은 보통 cafe_main iframe 안에 있음.
    """
    driver.switch_to.default_content()

    try:
        driver.switch_to.frame("cafe_main")
        return True
    except:
        return False


def get_soup():
    return BeautifulSoup(driver.page_source, "html.parser")


def safe_text(soup, selectors):
    """
    여러 selector 후보 중 먼저 잡히는 텍스트 반환
    """
    for selector in selectors:
        obj = soup.select_one(selector)
        if obj:
            txt = obj.get_text(" ", strip=True)
            if txt:
                return txt
    return ""


def safe_all_texts(soup, selectors):
    """
    여러 selector 후보에 해당하는 모든 텍스트 반환
    """
    results = []

    for selector in selectors:
        objs = soup.select(selector)

        for obj in objs:
            txt = obj.get_text(" ", strip=True)
            if txt:
                results.append(txt)

        if results:
            break

    return results

In [ ]:
# =====================================================
# 6. 목록 페이지에서 게시글 링크 수집
# =====================================================

def collect_article_links_from_current_page():
    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(" ", strip=True)

        # 제목 없는 링크 제거
        if not title or len(title) < 2:
            continue

        # /articles/숫자 형태만 허용
        if not re.search(r"/articles/\d+", href):
            continue

        # 상대경로 보정
        if href.startswith("/"):
            href = "https://cafe.naver.com" + href

        # 불필요한 파라미터 제거
        href = href.split("?")[0]
        # =====================================================
        # 목록에서 날짜 찾기
        # =====================================================

        list_date = None

        # 제목 링크 주변의 부모 영역에서 날짜 텍스트 탐색
        parent = a.find_parent()

        for _ in range(5):  # 부모를 최대 5단계까지 타고 올라가며 탐색
            if parent is None:
                break

            parent_text = parent.get_text(" ", strip=True)

            # 예: 2026.05.28. 13:24 / 05.28. 13:24 / 2026.05.28.
            date_match = re.search(
                r"(\d{4}\.\d{1,2}\.\d{1,2}\.)(?:\s+\d{1,2}:\d{2})?",
                parent_text
            )

            if date_match:
                list_date = parse_cafe_date(date_match.group(1))
                break

            parent = parent.find_parent()

        links.append({
            "url": href,
            "list_title": title,
            "list_date": list_date
        })

    # URL 기준 중복 제거
    unique = {}
    for item in links:
        unique[item["url"]] = item

    return list(unique.values())


def go_next_page():
    """
    목록 페이지에서 다음 페이지 클릭.
    실패하면 False 반환.
    """
    switch_to_cafe_iframe()

    next_selectors = [
        "a.pgR",
        "a.next",
        "a[aria-label='다음']"
    ]

    for selector in next_selectors:
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, selector)
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(SLEEP_SEC)
            return True
        except:
            pass

    return False

In [ ]:
# =====================================================
# 7. 게시글 본문 + 댓글 수집 함수
# =====================================================

def scrape_article(article_url):
    rows = []

    driver.get(article_url)
    time.sleep(SLEEP_SEC)

    switch_to_cafe_iframe()
    soup = get_soup()

    # 제목
    title = safe_text(soup, [
        "h3.title_text",
        ".title_text",
        ".tit-box span.b",
        ".article_header .title",
        "h3"
    ])

    # 작성일
    post_date_text = safe_text(soup, [
        ".date",
        ".article_info .date",
        ".info .date",
        "span.date"
    ])

    post_date = parse_cafe_date(post_date_text)

    # 본문
    body = safe_text(soup, [
        ".se-main-container",
        "#tbody",
        ".ContentRenderer",
        ".article_viewer",
        ".article_content"
    ])

    if body:
        rows.append({
            "source": "cafe",
            "round": "38R_before_after",
            "title": title,
            "post_date": post_date,
            "comment_date": None,
            "text_original": body,
            "text_clean": clean_text(body),
            "content_type": "post",
            "period": get_period(post_date),
            "url": article_url
        })

    # 댓글 수집
    comment_blocks = soup.select(".comment_box, .CommentItem, li.comment, .u_cbox_comment")

    if comment_blocks:
        for block in comment_blocks:
            comment_text = ""

            # 댓글 내용
            for selector in [
                ".text_comment",
                ".comment_text",
                ".u_cbox_contents",
                ".text",
                "span"
            ]:
                obj = block.select_one(selector)
                if obj:
                    comment_text = obj.get_text(" ", strip=True)
                    if comment_text:
                        break

            # 댓글 날짜
            comment_date_text = ""

            for selector in [
                ".comment_info_date",
                ".date",
                ".u_cbox_date",
                ".time"
            ]:
                obj = block.select_one(selector)
                if obj:
                    comment_date_text = obj.get_text(" ", strip=True)
                    if comment_date_text:
                        break

            comment_date = parse_cafe_date(comment_date_text)

            # 댓글 날짜가 없으면 게시글 날짜로 대체
            if comment_date is None:
                comment_date = post_date

            if comment_text:
                rows.append({
                    "source": "cafe",
                    "round": "38R_before_after",
                    "title": title,
                    "post_date": post_date,
                    "comment_date": comment_date,
                    "text_original": comment_text,
                    "text_clean": clean_text(comment_text),
                    "content_type": "comment",
                    "period": get_period(comment_date),
                    "url": article_url
                })

    return rows

## 1단계 링크 수집

In [ ]:
# =====================================================
# 8. 날짜 기준으로 게시글 링크 수집
#    - list_date 기준으로 START_DATE~END_DATE 범위만 저장
#    - 페이지가 START_DATE보다 과거로 넘어가면 중단
# =====================================================

all_article_links = []

for page in range(1, MAX_PAGES + 1):
    page_url = f"{BASE_URL}?page={page}"

    print(f"\n목록 {page}페이지 수집 중...")
    print(page_url)

    driver.get(page_url)
    time.sleep(SLEEP_SEC)

    links = collect_article_links_from_current_page()

    print(f"  - 게시글 링크 {len(links)}개 발견")

    if len(links) == 0:
        print("  - 링크가 없어 수집 중단")
        break

    # 날짜 확인용
    page_dates = [
        item["list_date"]
        for item in links
        if item.get("list_date") is not None
    ]

    # 날짜 범위에 들어오는 글만 후보 URL로 저장
    saved_this_page = 0

    for item in links:
        list_date = item.get("list_date")

        # 날짜가 없는 글은 일단 제외
        if list_date is None:
            continue

        if START_DATE <= list_date <= END_DATE:
            all_article_links.append(item)
            saved_this_page += 1

    print(f"  - 날짜 범위 내 링크 {saved_this_page}개 저장")

    if page_dates:
        newest = max(page_dates)
        oldest = min(page_dates)

        print(f"  - 페이지 최신 날짜: {newest}")
        print(f"  - 페이지 가장 오래된 날짜: {oldest}")

        # 최신 글도 START_DATE보다 과거면 이후 페이지는 더 볼 필요 없음
        if newest < START_DATE:
            print("  - 수집 시작일보다 과거 페이지에 도달하여 중단")
            break
    else:
        print("  - 이 페이지에서 날짜를 찾지 못함. 다음 페이지 계속 확인")

# 중복 제거
link_dict = {}

for item in all_article_links:
    link_dict[item["url"]] = item

all_article_links = list(link_dict.values())

# URL 후보 저장
article_links_df = pd.DataFrame(all_article_links)

article_links_df.to_csv(
    "cafe_article_links_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n최종 후보 게시글 링크 수:", len(all_article_links))
print("✅ 날짜 범위 URL 리스트 저장 완료")

display(article_links_df.head())

## 2단계 본문, 댓글 크롤링

In [ ]:
# =====================================================
# 9. 게시글별 본문/댓글 수집
# =====================================================

all_rows = []

for item in tqdm(all_article_links, desc="게시글 상세 수집"):
    url = item["url"]

    try:
        rows = scrape_article(url)

        # 날짜 범위 필터링
        for row in rows:
            기준일 = row["comment_date"] if row["content_type"] == "comment" else row["post_date"]

            if 기준일 is None:
                continue

            if START_DATE <= 기준일 <= END_DATE:
                all_rows.append(row)

    except Exception as e:
        print("수집 실패:", url, e)

    time.sleep(SLEEP_SEC)


cafe_df = pd.DataFrame(all_rows)

print("수집 완료:", cafe_df.shape)

display(cafe_df.head())

## 3단계 저장

In [ ]:
# =====================================================
# 10. 최종 정리 및 저장
# =====================================================

# 빈 텍스트 제거
cafe_df = cafe_df[cafe_df["text_clean"].notna()]
cafe_df = cafe_df[cafe_df["text_clean"].str.len() >= 2]

# 중복 제거
cafe_df = cafe_df.drop_duplicates(
    subset=["text_clean", "content_type", "url"]
)

# 날짜 기준 정렬
cafe_df = cafe_df.sort_values(
    by=["post_date", "comment_date"],
    ascending=True
)

# 저장
cafe_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

cafe_df.to_excel(
    OUTPUT_XLSX,
    index=False
)

print("최종 데이터 크기:", cafe_df.shape)
print("before/after 분포")
print(cafe_df["period"].value_counts())

print("content_type 분포")
print(cafe_df["content_type"].value_counts())

print("저장 완료")